# ERCOT RTM Price Spike Classifier — Classifier Notebook

Walk-forward CV (val 2022/2023/2024) and held-out test evaluation for binary spike classification (`RTM > $100/MWh`).
Loads parquet features from `data/processed/ercot/`; trains and saves classifier pkl files.

**Run time:** ~10–15 min (GARCH fits 3 folds for CV + test evaluation).

**Models evaluated:** Naive DAM>$100 baseline, XGB Classifier v1 (21 feat), v2 (29 feat), v3 (31 feat + GARCH vol); XGB Reg v3 used as classifier for comparison.
**Metric:** PR-AUC for model selection (continuous scores); F1 at fixed/optimal threshold for comparison.


In [1]:
# ── Standalone classifier setup — imports, helper functions, data load ─────────
from pathlib import Path
import numpy as np
import pandas as pd
import pickle
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    r2_score, mean_squared_error, mean_absolute_error,
    average_precision_score, f1_score, precision_score, recall_score,
    precision_recall_curve, brier_score_loss,
)
import xgboost as xgb
from arch import arch_model

PROC   = Path('data/processed/ercot')
TARGET = 'log_rtm_std'
TRAIN_END_FINAL = pd.Timestamp('2024-12-31 23:00')
Path('figures/modeling').mkdir(parents=True, exist_ok=True)

# ── Helper functions (same as modeling.ipynb §7) ──────────────────────────────
def make_seasonal(df):
    tmp = pd.DataFrame({
        'hour':  df.index.hour,
        'month': df.index.month,
        'dow':   df.index.dayofweek,
    }, index=df.index)
    return pd.get_dummies(tmp.astype(str), drop_first=True)

def build_garch_vol(df, train_end):
    import warnings
    target = df[TARGET] if TARGET in df.columns else df.iloc[:, 0]
    tr_mask = target.index <= train_end
    S_tr = make_seasonal(target[tr_mask].to_frame())
    seas_ols = LinearRegression().fit(S_tr, target[tr_mask])
    S_all = make_seasonal(target.to_frame()).reindex(columns=S_tr.columns, fill_value=0)
    resid = target - seas_ols.predict(S_all)
    split = int(tr_mask.sum())
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        gm = arch_model(resid, mean='Constant', vol='GARCH', p=1, q=1, dist='t')
        gr = gm.fit(last_obs=split, disp='off', show_warning=False)
    fc = gr.forecast(horizon=1, start=0, reindex=False)
    cond_vol = pd.Series(
        np.sqrt(np.clip(fc.variance['h.1'].values, 0, None)),
        index=target.index
    )
    return cond_vol.shift(24)   # D-1 lag — leakage-safe

def add_engineered_features(df):
    df = df.copy()
    df['fc_net_load']          = df['fc_coast'] - df['wf_stwpf_lz_south_houston']
    df['dam_rtm_spread']       = df['dam_price_houston'] - df['rtm_mean_lag24']
    df['abs_dam_rtm_spread']   = df['dam_rtm_spread'].abs()
    df['week']                 = df.index.isocalendar().week.astype(int)
    df['load_lag7d']           = df['load_houston_lag48'].shift(168)
    df['rtm_price_std_lag7d']  = df['rtm_std_lag24'].shift(168)
    df['rtm_price_mean_lag7d'] = df['rtm_mean_lag24'].shift(168)
    df['outage_fraction']      = df['total_resource_mw'] / (df['fc_system_total'] + 1)
    df['hour']                 = df.index.hour
    df['month']                = df.index.month
    df['dow']                  = df.index.dayofweek
    return df

# ── Load and combine parquet features ────────────────────────────────────────
train_df = pd.read_parquet(PROC / 'train_features.parquet')
test_df  = pd.read_parquet(PROC / 'test_features.parquet')
combined = pd.concat([train_df, test_df]).sort_index()
combined = add_engineered_features(combined)

# ── Imputation (same strategy as modeling.ipynb) ─────────────────────────────
_train_mask = combined.index <= TRAIN_END_FINAL
for _col in ['rtm_std_lag24', 'load_houston_lag48']:
    if _col in combined.columns:
        combined[_col] = combined[_col].ffill()
for _col in ['dam_price_houston', 'system_lambda',
             'mcpc_regup', 'mcpc_rrs', 'mcpc_nspin', 'mcpc_regdn']:
    if _col in combined.columns:
        _med = combined.loc[_train_mask, _col].median()
        combined[_col] = combined[_col].fillna(_med)
if 'total_resource_mw' in combined.columns:
    combined['total_resource_mw'] = combined['total_resource_mw'].fillna(0)

print(f"Data loaded: {len(combined)} rows, {combined.index[0].date()} → {combined.index[-1].date()}")


Data loaded: 74472 rows, 2017-07-04 → 2025-12-31


In [ ]:

# ── Classifier Error Analysis & Model Leaderboard ───────────────────────────
# Requires: add_engineered_features, build_garch_vol from Section 7.3
# Loads all model artifacts from pkl files — no dependency on in-memory vars

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from sklearn.metrics import (r2_score, mean_squared_error, mean_absolute_error,
                             average_precision_score,
                             brier_score_loss, f1_score)
import pickle
from pathlib import Path

PROC   = Path('data/processed/ercot')
TARGET = 'log_rtm_std'

# ── Load best model artifacts ─────────────────────────────────────────────────
with open(PROC / 'model_xgb_reg_v3_tuned.pkl', 'rb') as f:   # PRIMARY: tuned (depth=4, lr=0.03)
    m_v3 = pickle.load(f)
with open(PROC / 'model_xgb_clf_v3.pkl', 'rb') as f:
    m_clf = pickle.load(f)
with open(PROC / 'model_xgb_reg_v2.pkl', 'rb') as f:
    m_v2 = pickle.load(f)
with open(PROC / 'model_xgb_reg_v2_xgb.pkl', 'rb') as f:
    m_v2_f = pickle.load(f)

# Feature name fallbacks (for models trained with numpy arrays — feature_names=None)
_FEAT_V2_FALLBACK = [
    'dam_price_houston',
    'load_houston_lag48', 'rtm_mean_lag24', 'rtm_std_lag24',
    'total_resource_mw',
    'fc_system_total', 'wgrpp_lz_south_houston', 'wind_error_houston',
    'fc_coast', 'wf_stwpf_lz_south_houston',
    'temp_f_houston_avg', 'humidity_pct_houston_avg',
    'wind_gust_mph_houston_avg', 'precip_in_houston_avg',
    'mcpc_regup', 'mcpc_rrs', 'mcpc_nspin', 'mcpc_regdn',
    'hour', 'month', 'dow',
    'fc_net_load', 'dam_rtm_spread', 'week',
    'load_lag7d', 'rtm_price_std_lag7d', 'rtm_price_mean_lag7d', 'outage_fraction',
    'abs_dam_rtm_spread',
]  # 29 features (FEAT_V2)
_FEAT_V3_FALLBACK = _FEAT_V2_FALLBACK + ['system_lambda', 'garch_cond_vol']  # 31 features

_fn_v3 = m_v3.get_booster().feature_names
FEAT_V3 = list(_fn_v3) if _fn_v3 is not None else _FEAT_V3_FALLBACK

_fn_v2 = m_v2.get_booster().feature_names
FEAT_V2_names = list(_fn_v2) if _fn_v2 is not None else _FEAT_V2_FALLBACK

# ── Rebuild combined feature matrix with GARCH vol ───────────────────────────
train_df = pd.read_parquet(PROC / 'train_features.parquet')
test_df  = pd.read_parquet(PROC / 'test_features.parquet')
combined = pd.concat([train_df, test_df]).sort_index()
combined = add_engineered_features(combined)

TRAIN_END_FINAL = pd.Timestamp('2024-12-31 23:00')

# ── Column-specific imputation (fit on train only) ────────────────────────────
_train_mask_s8 = combined.index <= TRAIN_END_FINAL

# ffill for time-series lag features
for _col in ['rtm_std_lag24', 'load_houston_lag48']:
    if _col in combined.columns:
        combined[_col] = combined[_col].ffill()

# Median imputation for price/market features (fit on train, apply to both)
_price_cols = ['dam_price_houston', 'system_lambda',
               'mcpc_regup', 'mcpc_rrs', 'mcpc_nspin', 'mcpc_regdn']
for _col in _price_cols:
    if _col in combined.columns:
        _med = combined.loc[_train_mask_s8, _col].median()
        combined[_col] = combined[_col].fillna(_med)

# total_resource_mw: keep fillna(0) — nulls are pre-2022, final model unaffected
if 'total_resource_mw' in combined.columns:
    combined['total_resource_mw'] = combined['total_resource_mw'].fillna(0)

garch_vol_s8 = build_garch_vol(combined[[TARGET]], TRAIN_END_FINAL)
combined['garch_cond_vol'] = garch_vol_s8

# ── Test slice ────────────────────────────────────────────────────────────────
test  = combined[combined.index > TRAIN_END_FINAL].copy()
y_te  = test[TARGET]

# ── Regression predictions — XGBoost handles remaining NaN natively ───────────
X_te_v3 = test[FEAT_V3]
pred_v3  = m_v3.predict(X_te_v3)
resid    = y_te.values - pred_v3

# ── Spike mask and classifier ─────────────────────────────────────────────────
spike_mask = test['spike_flag'].astype(bool)
y_te_clf   = test['spike_flag'].astype(int)

_fn_clf    = m_clf.get_booster().feature_names
_clf_feats = list(_fn_clf) if _fn_clf is not None else FEAT_V3
prob_clf   = m_clf.predict_proba(test[_clf_feats])[:, 1]

print(f"§8 setup complete.")
print(f"  Test: {len(test)} rows  |  {test.index[0].date()} → {test.index[-1].date()}")
print(f"  XGB v3 tuned  R²={r2_score(y_te, pred_v3):.3f}  RMSE={mean_squared_error(y_te, pred_v3)**0.5:.4f}  MAE(orig)={mean_absolute_error(np.expm1(y_te), np.expm1(pred_v3)):.3f}")
print(f"  Spike rate: {spike_mask.mean():.1%}")


§8 setup complete.
  Test: 8760 rows  |  2025-01-01 → 2025-12-31
  XGB v3 tuned  R²=0.386  RMSE=0.6687  MAE(orig)=4.027
  Spike rate: 2.2%


In [3]:
# ── Walk-forward CV for XGB Spike Classifiers (v1, v2, v3) ───────────────────
# Uses PR-AUC as selection metric — appropriate for 2.2% spike rate.
# 3 folds: val=2022, 2023, 2024 (expanding training window, same as regression CV).
# Pools val predictions across folds for CV diagnostic plots.
# Reuses _garch_cache from §7.3.1 grid search cell for v3 GARCH vol (no refitting).
# Requires: add_engineered_features, build_garch_vol from §7.3.

import numpy as np
import pandas as pd
import pickle
import xgboost as xgb
from sklearn.metrics import (average_precision_score, precision_recall_curve,
                             f1_score, precision_score, recall_score)
from pathlib import Path

_PROC_CLF = Path('data/processed/ercot')
_TARGET_CLF = 'log_rtm_std'

# ── Load and prepare data ─────────────────────────────────────────────────────
_tr_cv = pd.read_parquet(_PROC_CLF / 'train_features.parquet')
_te_cv = pd.read_parquet(_PROC_CLF / 'test_features.parquet')
_comb_cv = pd.concat([_tr_cv, _te_cv]).sort_index()
_comb_cv = add_engineered_features(_comb_cv)

_TRAIN_END_CV = pd.Timestamp('2024-12-31 23:00')
_tr_mask_cv   = _comb_cv.index <= _TRAIN_END_CV
for _col in ['rtm_std_lag24', 'load_houston_lag48']:
    if _col in _comb_cv.columns:
        _comb_cv[_col] = _comb_cv[_col].ffill()
for _col in ['dam_price_houston', 'system_lambda',
             'mcpc_regup', 'mcpc_rrs', 'mcpc_nspin', 'mcpc_regdn']:
    if _col in _comb_cv.columns:
        _med = _comb_cv.loc[_tr_mask_cv, _col].median()
        _comb_cv[_col] = _comb_cv[_col].fillna(_med)
if 'total_resource_mw' in _comb_cv.columns:
    _comb_cv['total_resource_mw'] = _comb_cv['total_resource_mw'].fillna(0)

# ── Feature sets ──────────────────────────────────────────────────────────────
_F_V1 = [
    'dam_price_houston', 'load_houston_lag48', 'rtm_mean_lag24', 'rtm_std_lag24',
    'total_resource_mw', 'fc_system_total', 'wgrpp_lz_south_houston', 'wind_error_houston',
    'fc_coast', 'wf_stwpf_lz_south_houston', 'temp_f_houston_avg', 'humidity_pct_houston_avg',
    'wind_gust_mph_houston_avg', 'precip_in_houston_avg',
    'mcpc_regup', 'mcpc_rrs', 'mcpc_nspin', 'mcpc_regdn', 'hour', 'month', 'dow',
]  # 21
_F_V2 = _F_V1 + [
    'fc_net_load', 'dam_rtm_spread', 'week',
    'load_lag7d', 'rtm_price_std_lag7d', 'rtm_price_mean_lag7d',
    'outage_fraction', 'abs_dam_rtm_spread',
]  # 29
_F_V2_XGB = _F_V2 + ['system_lambda']   # 30
_F_V3     = _F_V2_XGB + ['garch_cond_vol']  # 31

_CLF_PARAMS = {
    'v1': dict(n_estimators=500, max_depth=6, learning_rate=0.05,
               subsample=0.8, colsample_bytree=0.8,
               random_state=42, n_jobs=-1, verbosity=0, eval_metric='logloss'),
    'v2': dict(n_estimators=800, max_depth=5, learning_rate=0.04,
               subsample=0.8, colsample_bytree=0.8, min_child_weight=3, gamma=0.1,
               random_state=42, n_jobs=-1, verbosity=0, eval_metric='logloss'),
    'v3': dict(n_estimators=600, max_depth=5, learning_rate=0.05,
               subsample=0.8, colsample_bytree=0.8, min_child_weight=3,
               random_state=42, n_jobs=-1, verbosity=0, eval_metric='logloss'),
}

_clf_fold_ends  = [pd.Timestamp('2021-12-31 23:00'),
                   pd.Timestamp('2022-12-31 23:00'),
                   pd.Timestamp('2023-12-31 23:00')]
_clf_fold_names = ['2022', '2023', '2024']

# ── Collectors ────────────────────────────────────────────────────────────────
# naive_threshold: F1 at fixed $100 (single operating point — comparable to XGB F1)
# naive_continuous: DAM price as continuous score → PR-AUC only (no fixed threshold)
# XGB: PR-AUC for model selection + F1 at fold-optimal threshold for comparison
_clf_cv_prauc  = {'v1': [], 'v2': [], 'v3': []}        # XGB PR-AUC per fold
_clf_cv_f1     = {'naive': [], 'v1': [], 'v2': [], 'v3': []}  # F1 per fold
_clf_cv_probs  = {'v1': [], 'v2': [], 'v3': []}         # pooled probs for plots
_clf_cv_labels = []                                      # pooled labels for plots
_clf_cv_dam    = []                                      # DAM continuous scores for plots

for _fe, _fn in zip(_clf_fold_ends, _clf_fold_names):
    _val_start = _fe + pd.Timedelta(hours=1)
    _val_end   = _fe + pd.DateOffset(years=1)
    _tr_f  = _comb_cv.loc[_comb_cv.index <= _fe].dropna(subset=[_TARGET_CLF, 'spike_flag'])
    _va_f  = _comb_cv.loc[(_comb_cv.index >= _val_start) & (_comb_cv.index <= _val_end)].dropna(subset=[_TARGET_CLF, 'spike_flag'])

    _y_tr_c = _tr_f['spike_flag'].astype(int)
    _y_va_c = _va_f['spike_flag'].astype(int)
    _spw    = (1 - _y_tr_c.mean()) / _y_tr_c.mean()
    _clf_cv_labels.append(_y_va_c.values)

    # ── Naive: fixed $100 threshold → F1 only ────────────────────────────────
    _dam_scores_fold = _va_f['dam_price_houston'].fillna(0).values
    _clf_cv_dam.append(_dam_scores_fold)
    _naive_pred = (_dam_scores_fold >= 100).astype(int)
    _clf_cv_f1['naive'].append(f1_score(_y_va_c, _naive_pred, zero_division=0))

    print(f"Fold {_fn}: train={len(_tr_f):,}  val={len(_va_f):,}  "
          f"spike_rate_tr={_y_tr_c.mean():.2%}  spike_rate_val={_y_va_c.mean():.2%}")

    # v1
    _m = xgb.XGBClassifier(**_CLF_PARAMS['v1'], scale_pos_weight=_spw)
    _m.fit(_tr_f[_F_V1].values, _y_tr_c)
    _p = _m.predict_proba(_va_f[_F_V1].values)[:, 1]
    _clf_cv_probs['v1'].append(_p)
    _clf_cv_prauc['v1'].append(average_precision_score(_y_va_c, _p))
    _pr, _rc, _th = precision_recall_curve(_y_va_c, _p)
    _f1s = 2 * _pr[:-1] * _rc[:-1] / (_pr[:-1] + _rc[:-1] + 1e-9)
    _clf_cv_f1['v1'].append(_f1s.max())

    # v2
    _m = xgb.XGBClassifier(**_CLF_PARAMS['v2'], scale_pos_weight=_spw)
    _m.fit(_tr_f[_F_V2].values, _y_tr_c)
    _p = _m.predict_proba(_va_f[_F_V2].values)[:, 1]
    _clf_cv_probs['v2'].append(_p)
    _clf_cv_prauc['v2'].append(average_precision_score(_y_va_c, _p))
    _pr, _rc, _th = precision_recall_curve(_y_va_c, _p)
    _f1s = 2 * _pr[:-1] * _rc[:-1] / (_pr[:-1] + _rc[:-1] + 1e-9)
    _clf_cv_f1['v2'].append(_f1s.max())

    # v3 (needs garch_cond_vol)
    try:
        _gv = _garch_cache[_fn]
    except NameError:
        print(f"  _garch_cache not found — rebuilding GARCH for fold {_fn} (~2 min)...")
        _gv = build_garch_vol(_comb_cv[[_TARGET_CLF]], _fe)
    _cv3 = _comb_cv.copy()
    _cv3['garch_cond_vol'] = _gv
    _tr_v3 = _cv3.loc[_cv3.index <= _fe].dropna(subset=[_TARGET_CLF, 'spike_flag'])
    _va_v3 = _cv3.loc[(_cv3.index >= _val_start) & (_cv3.index <= _val_end)].dropna(subset=[_TARGET_CLF, 'spike_flag'])
    _m = xgb.XGBClassifier(**_CLF_PARAMS['v3'], scale_pos_weight=_spw)
    _m.fit(_tr_v3[_F_V3].values, _tr_v3['spike_flag'].astype(int))
    _p = _m.predict_proba(_va_v3[_F_V3].values)[:, 1]
    _clf_cv_probs['v3'].append(_p)
    _clf_cv_prauc['v3'].append(average_precision_score(_va_v3['spike_flag'].astype(int), _p))
    _pr, _rc, _th = precision_recall_curve(_va_v3['spike_flag'].astype(int), _p)
    _f1s = 2 * _pr[:-1] * _rc[:-1] / (_pr[:-1] + _rc[:-1] + 1e-9)
    _clf_cv_f1['v3'].append(_f1s.max())

    print(f"  naive F1(t=$100)={_clf_cv_f1['naive'][-1]:.3f}  "
          f"v1 PR-AUC={_clf_cv_prauc['v1'][-1]:.3f} F1={_clf_cv_f1['v1'][-1]:.3f}  "
          f"v2 PR-AUC={_clf_cv_prauc['v2'][-1]:.3f} F1={_clf_cv_f1['v2'][-1]:.3f}  "
          f"v3 PR-AUC={_clf_cv_prauc['v3'][-1]:.3f} F1={_clf_cv_f1['v3'][-1]:.3f}")

# ── Pool predictions across folds ─────────────────────────────────────────────
_y_cv_pooled   = np.concatenate(_clf_cv_labels)
_prob_v1_cv    = np.concatenate(_clf_cv_probs['v1'])
_prob_v2_cv    = np.concatenate(_clf_cv_probs['v2'])
_prob_v3_cv    = np.concatenate(_clf_cv_probs['v3'])
_dam_cv_scores = np.concatenate(_clf_cv_dam)   # DAM continuous score (for PR curve plot)

# ── Summary tables ────────────────────────────────────────────────────────────
print("\n── Naive (fixed $100 threshold) — CV F1 ──")
print(f"{'Model':<26} {'Mean F1':>8}  {'2022':>7}  {'2023':>7}  {'2024':>7}  Selection")
print("-" * 72)
_s = _clf_cv_f1['naive']
print(f"{'Naive: DAM > $100':<26} {np.mean(_s):>8.3f}  {_s[0]:>7.3f}  {_s[1]:>7.3f}  {_s[2]:>7.3f}  ✅ BEST (fixed t)")

print("\n── XGB Classifiers — CV PR-AUC (model selection) + best-threshold F1 ──")
print(f"{'Model':<26} {'PR-AUC':>8}  {'2022':>7}  {'2023':>7}  {'2024':>7}  "
      f"{'F1 mean':>8}  {'2022':>7}  {'2023':>7}  {'2024':>7}  Selection")
print("-" * 105)
_best_clf = max(_clf_cv_prauc, key=lambda v: np.mean(_clf_cv_prauc[v]))
for _v, _fc in [('v1', 21), ('v2', 29), ('v3', 31)]:
    _pa = _clf_cv_prauc[_v]
    _f1 = _clf_cv_f1[_v]
    _sel = '✅ SELECTED' if _v == _best_clf else ''
    print(f"{'XGB Clf ' + _v + ' (' + str(_fc) + ' feat)':<26} {np.mean(_pa):>8.3f}  "
          f"{_pa[0]:>7.3f}  {_pa[1]:>7.3f}  {_pa[2]:>7.3f}  "
          f"{np.mean(_f1):>8.3f}  {_f1[0]:>7.3f}  {_f1[1]:>7.3f}  {_f1[2]:>7.3f}  {_sel}")

print(f"\nPooled CV spike rate: {_y_cv_pooled.mean():.2%}  ({_y_cv_pooled.sum()} / {len(_y_cv_pooled)} hours)")
_prauc_naive_cv = average_precision_score(_y_cv_pooled, _dam_cv_scores)
print(f"DAM continuous score pooled CV PR-AUC: {_prauc_naive_cv:.3f}  (upper bound — no fixed threshold)")
print("Pooled CV predictions: _prob_v1_cv / _prob_v2_cv / _prob_v3_cv / _dam_cv_scores + _y_cv_pooled")

# ── Train final classifiers on full train set and save pkl artifacts ──────────
print("\n── Training final classifiers on full train (2017–2024) ──")
_full_train = _comb_cv.loc[_comb_cv.index <= _TRAIN_END_CV].dropna(subset=[_TARGET_CLF, 'spike_flag'])
_full_test  = _comb_cv.loc[_comb_cv.index > _TRAIN_END_CV].dropna(subset=[_TARGET_CLF, 'spike_flag'])
_y_full_tr  = _full_train['spike_flag'].astype(int)
_y_full_te  = _full_test['spike_flag'].astype(int)
_spw_full   = (1 - _y_full_tr.mean()) / _y_full_tr.mean()

_clf_v1_final = xgb.XGBClassifier(**_CLF_PARAMS['v1'], scale_pos_weight=_spw_full)
_clf_v1_final.fit(_full_train[_F_V1], _y_full_tr)
with open(_PROC_CLF / 'model_xgb_clf.pkl', 'wb') as f:
    pickle.dump(_clf_v1_final, f)

_clf_v2_final = xgb.XGBClassifier(**_CLF_PARAMS['v2'], scale_pos_weight=_spw_full)
_clf_v2_final.fit(_full_train[_F_V2], _y_full_tr)
with open(_PROC_CLF / 'model_xgb_clf_v2.pkl', 'wb') as f:
    pickle.dump(_clf_v2_final, f)

_gv_final = build_garch_vol(_comb_cv[[_TARGET_CLF]], _TRAIN_END_CV)
_comb_v3_final = _comb_cv.copy()
_comb_v3_final['garch_cond_vol'] = _gv_final
_tr_v3_final = _comb_v3_final.loc[_comb_v3_final.index <= _TRAIN_END_CV].dropna(subset=[_TARGET_CLF, 'spike_flag'])
_clf_v3_final = xgb.XGBClassifier(**_CLF_PARAMS['v3'], scale_pos_weight=_spw_full)
_clf_v3_final.fit(_tr_v3_final[_F_V3], _tr_v3_final['spike_flag'].astype(int))
with open(_PROC_CLF / 'model_xgb_clf_v3.pkl', 'wb') as f:
    pickle.dump(_clf_v3_final, f)

# ── Test-set metrics ──────────────────────────────────────────────────────────
_te_v3_final = _comb_v3_final.loc[_comb_v3_final.index > _TRAIN_END_CV].dropna(subset=[_TARGET_CLF, 'spike_flag'])

print(f"\n── Final Classifier Test Metrics (2025) ──")
print(f"{'Model':<30} {'PR-AUC':>8} {'F1':>6} {'Prec':>6} {'Recall':>8} {'Threshold':>10}")
print("-" * 72)

# Naive threshold
_dam_te = _full_test['dam_price_houston'].fillna(0).values
_y_te_c = _full_test['spike_flag'].astype(int)
_naive_te_pred = (_dam_te >= 100).astype(int)
_naive_prauc = average_precision_score(_y_te_c, _dam_te)   # continuous score
_naive_f1    = f1_score(_y_te_c, _naive_te_pred, zero_division=0)
_naive_prec  = precision_score(_y_te_c, _naive_te_pred, zero_division=0)
_naive_rec   = recall_score(_y_te_c, _naive_te_pred, zero_division=0)
print(f"{'Naive: DAM > $100 (fixed t)':<30} {'—':>8} {_naive_f1:>6.3f} {_naive_prec:>6.3f} {_naive_rec:>8.3f} {'$100':>10}")
print(f"{'DAM continuous score':<30} {_naive_prauc:>8.3f} {'—':>6} {'—':>6} {'—':>8} {'swept':>10}")

for _name, _model, _feats, _te_df in [
    ('XGB Clf v1 (21)', _clf_v1_final, _F_V1, _full_test),
    ('XGB Clf v2 (29)', _clf_v2_final, _F_V2, _full_test),
    ('XGB Clf v3 (31)', _clf_v3_final, _F_V3, _te_v3_final),
]:
    _prob = _model.predict_proba(_te_df[_feats])[:, 1]
    _y_te = _te_df['spike_flag'].astype(int)
    _prauc = average_precision_score(_y_te, _prob)
    _pr, _rc, _th = precision_recall_curve(_y_te, _prob)
    _f1_arr = 2 * _pr[:-1] * _rc[:-1] / (_pr[:-1] + _rc[:-1] + 1e-9)
    _best = np.argmax(_f1_arr)
    _opt_t = _th[_best]
    _pred = (_prob >= _opt_t).astype(int)
    _f1 = f1_score(_y_te, _pred)
    _prec = precision_score(_y_te, _pred)
    _rec = recall_score(_y_te, _pred)
    print(f"{_name:<30} {_prauc:>8.3f} {_f1:>6.3f} {_prec:>6.3f} {_rec:>8.3f} {_opt_t:>10.3f}")

print(f"\nSaved: model_xgb_clf.pkl, model_xgb_clf_v2.pkl, model_xgb_clf_v3.pkl")

Fold 2022: train=39,408  val=8,760  spike_rate_tr=2.51%  spike_rate_val=7.29%


  _garch_cache not found — rebuilding GARCH for fold 2022 (~2 min)...


  naive F1(t=$100)=0.467  v1 PR-AUC=0.221 F1=0.314  v2 PR-AUC=0.226 F1=0.319  v3 PR-AUC=0.225 F1=0.301
Fold 2023: train=48,168  val=8,760  spike_rate_tr=3.38%  spike_rate_val=4.11%


  _garch_cache not found — rebuilding GARCH for fold 2023 (~2 min)...


  naive F1(t=$100)=0.518  v1 PR-AUC=0.486 F1=0.547  v2 PR-AUC=0.493 F1=0.542  v3 PR-AUC=0.473 F1=0.546
Fold 2024: train=56,928  val=8,784  spike_rate_tr=3.49%  spike_rate_val=1.71%


  _garch_cache not found — rebuilding GARCH for fold 2024 (~2 min)...


  naive F1(t=$100)=0.428  v1 PR-AUC=0.380 F1=0.438  v2 PR-AUC=0.359 F1=0.452  v3 PR-AUC=0.354 F1=0.430

── Naive (fixed $100 threshold) — CV F1 ──
Model                       Mean F1     2022     2023     2024  Selection
------------------------------------------------------------------------
Naive: DAM > $100             0.471    0.467    0.518    0.428  ✅ BEST (fixed t)

── XGB Classifiers — CV PR-AUC (model selection) + best-threshold F1 ──
Model                        PR-AUC     2022     2023     2024   F1 mean     2022     2023     2024  Selection
---------------------------------------------------------------------------------------------------------
XGB Clf v1 (21 feat)          0.363    0.221    0.486    0.380     0.433    0.314    0.547    0.438  ✅ SELECTED
XGB Clf v2 (29 feat)          0.359    0.226    0.493    0.359     0.438    0.319    0.542    0.452  
XGB Clf v3 (31 feat)          0.351    0.225    0.473    0.354     0.426    0.301    0.546    0.430  

Pooled CV spike ra


── Final Classifier Test Metrics (2025) ──
Model                            PR-AUC     F1   Prec   Recall  Threshold
------------------------------------------------------------------------
Naive: DAM > $100 (fixed t)           —  0.294  0.366    0.245       $100
DAM continuous score              0.296      —      —        —      swept
XGB Clf v1 (21)                   0.229  0.323  0.239    0.500      0.233


XGB Clf v2 (29)                   0.237  0.320  0.251    0.439      0.417
XGB Clf v3 (31)                   0.237  0.329  0.251    0.474      0.419

Saved: model_xgb_clf.pkl, model_xgb_clf_v2.pkl, model_xgb_clf_v3.pkl


In [ ]:

# ── CV — Classifier Diagnostics (Pooled Walk-forward CV, 2022–2024) ─────
# Requires: _prob_v1_cv, _prob_v2_cv, _prob_v3_cv, _y_cv_pooled, _dam_cv_scores
# Per-fold F1: _clf_cv_f1 dict from 8plveypungu-clf

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import (precision_recall_curve, average_precision_score,
                             f1_score, precision_score, recall_score,
                             confusion_matrix, ConfusionMatrixDisplay)

_cv_baseline = _y_cv_pooled.mean()

_cv_models = [
    ('XGB Clf v1 (21 feat)', _prob_v1_cv, 'tab:blue'),
    ('XGB Clf v2 (29 feat)', _prob_v2_cv, 'tab:green'),
    ('XGB Clf v3 (31 feat)', _prob_v3_cv, 'tab:red'),
]

# ── Naive $100 point (CV pooled) ──────────────────────────────────────────────
_naive_pred_cv = (_dam_cv_scores >= 100).astype(int)
_naive_prec_cv = precision_score(_y_cv_pooled, _naive_pred_cv, zero_division=0)
_naive_rec_cv  = recall_score(_y_cv_pooled, _naive_pred_cv, zero_division=0)
_naive_f1_cv   = f1_score(_y_cv_pooled, _naive_pred_cv, zero_division=0)

# ── CV Diagnostics 2×3 ────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('XGBoost Classifiers — CV Diagnostics (Pooled Walk-forward Val 2022–2024)', fontsize=12)

# (0,0) PR curves overlaid
ax = axes[0, 0]
for name, probs, color in _cv_models:
    prec, rec, _ = precision_recall_curve(_y_cv_pooled, probs)
    ap = average_precision_score(_y_cv_pooled, probs)
    ax.plot(rec, prec, color=color, lw=2, label=f'{name}  PR-AUC={ap:.3f}')
ax.axhline(_cv_baseline, color='gray', ls=':', lw=1, label=f'Random ({_cv_baseline:.3f})')
ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
ax.set_title('PR Curves — CV (pooled val)'); ax.legend(fontsize=8)

# (0,1) F1 vs threshold
ax = axes[0, 1]
for name, probs, color in _cv_models:
    prec, rec, thresh = precision_recall_curve(_y_cv_pooled, probs)
    f1 = 2 * prec[:-1] * rec[:-1] / (prec[:-1] + rec[:-1] + 1e-9)
    best_i = np.argmax(f1)
    ax.plot(thresh, f1, color=color, lw=1.5, alpha=0.8,
            label=f'{name.split("(")[0].strip()}  F1={f1[best_i]:.3f}')
    ax.scatter(thresh[best_i], f1[best_i], color=color, s=40, zorder=5)
ax.set_xlabel('Threshold'); ax.set_ylabel('F1 Score')
ax.set_title('F1 vs Threshold — CV'); ax.legend(fontsize=8)

# (0,2) PR-AUC bar chart
ax = axes[0, 2]
_cv_names  = ['XGB\nClf v1', 'XGB\nClf v2', 'XGB\nClf v3']
_cv_colors = ['tab:blue', 'tab:green', 'tab:red']
_cv_prs    = [average_precision_score(_y_cv_pooled, p) for _, p, _ in _cv_models]
bars = ax.bar(_cv_names, _cv_prs, color=_cv_colors, alpha=0.85, edgecolor='gray')
ax.axhline(_cv_baseline, color='k', ls=':', lw=1, label=f'Random ({_cv_baseline:.3f})')
for bar, val in zip(bars, _cv_prs):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.003, f'{val:.3f}',
            ha='center', fontsize=10, fontweight='bold')
ax.set_ylabel('PR-AUC'); ax.set_title('PR-AUC — CV'); ax.legend(fontsize=8)

# (1,0) Gains chart
ax = axes[1, 0]
for name, probs, color in _cv_models:
    _idx = np.argsort(probs)[::-1]
    _gains = np.cumsum(_y_cv_pooled[_idx]) / _y_cv_pooled.sum()
    _pct   = np.arange(1, len(_y_cv_pooled) + 1) / len(_y_cv_pooled)
    ax.plot(_pct, _gains, color=color, lw=1.8, label=name.split('(')[0].strip())
ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Random')
ax.plot([0, _cv_baseline, 1], [0, 1, 1], 'g--', lw=1, label='Perfect')
ax.set_xlabel('% Population (by score)'); ax.set_ylabel('% Spikes Captured')
ax.set_title('Gains Chart — CV'); ax.legend(fontsize=8)

# (1,1) Confusion matrix — v3 at optimal CV threshold
ax = axes[1, 1]
_prec_v3, _rec_v3, _thr_v3 = precision_recall_curve(_y_cv_pooled, _prob_v3_cv)
_f1_v3 = 2 * _prec_v3[:-1] * _rec_v3[:-1] / (_prec_v3[:-1] + _rec_v3[:-1] + 1e-9)
_opt_v3 = np.argmax(_f1_v3)
_thr_opt_v3_cv = _thr_v3[_opt_v3]
_pred_cv_v3 = (_prob_v3_cv >= _thr_opt_v3_cv).astype(int)
_cm_cv = confusion_matrix(_y_cv_pooled, _pred_cv_v3)
ConfusionMatrixDisplay(_cm_cv, display_labels=['Non-spike', 'Spike']).plot(
    ax=ax, colorbar=False, cmap='Blues')
ax.set_title(f'Confusion Matrix — CV v3 (t={_thr_opt_v3_cv:.3f})')

# (1,2) F1 bar chart — pooled CV optimal F1
ax = axes[1, 2]
_cv_f1s = []
for _, probs, _ in _cv_models:
    prec, rec, _ = precision_recall_curve(_y_cv_pooled, probs)
    f1 = 2 * prec[:-1] * rec[:-1] / (prec[:-1] + rec[:-1] + 1e-9)
    _cv_f1s.append(f1.max())
bars = ax.bar(_cv_names, _cv_f1s, color=_cv_colors, alpha=0.85, edgecolor='gray')
for bar, val in zip(bars, _cv_f1s):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.003, f'{val:.3f}',
            ha='center', fontsize=10, fontweight='bold')
ax.set_ylabel('F1 (at optimal threshold)'); ax.set_title('F1 — CV')

plt.tight_layout()
plt.savefig('figures/modeling/classification_diagnostics_cv.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved figures/modeling/classification_diagnostics_cv.png")

# ── PR Curve Comparison — 2×2 with two naive baselines ───────────────────────
from sklearn.preprocessing import minmax_scale as _mms

_xgb_cv_clfs = [
    ('XGB Classifier v1 (21 feat)', _prob_v1_cv, 'tab:blue',  '-'),
    ('XGB Classifier v2 (29 feat)', _prob_v2_cv, 'tab:green', '-'),
    ('XGB Classifier v3 (31 feat)', _prob_v3_cv, 'tab:red',   '-'),
]

_fig_pr_cv, _axes_pr_cv = plt.subplots(2, 2, figsize=(12, 10))
_fig_pr_cv.suptitle('Precision-Recall Curves — All Spike Classifiers (CV pooled val 2022–2024)',
                     fontsize=13)

# (0,0) PR curves: DAM continuous + Naive $100 star + XGB curves
ax = _axes_pr_cv[0, 0]
_dam_prec_cv, _dam_rec_cv, _ = precision_recall_curve(_y_cv_pooled, _dam_cv_scores)
_dam_ap_cv = average_precision_score(_y_cv_pooled, _dam_cv_scores)
ax.plot(_dam_rec_cv, _dam_prec_cv, color='gray', ls='--', lw=2,
        label=f'DAM continuous score (PR-AUC={_dam_ap_cv:.3f})')
ax.scatter(_naive_rec_cv, _naive_prec_cv, color='gray', marker='*', s=250, zorder=6,
           label=f'Naive: DAM>$100 (F1={_naive_f1_cv:.3f}, point only)')
for name, probs, color, ls in _xgb_cv_clfs:
    prec, rec, _ = precision_recall_curve(_y_cv_pooled, probs)
    ap = average_precision_score(_y_cv_pooled, probs)
    ax.plot(rec, prec, color=color, ls=ls, lw=2, label=f'{name} (PR-AUC={ap:.3f})')
ax.axhline(_cv_baseline, color='black', ls=':', lw=1, label=f'Random ({_cv_baseline:.3f})')
ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
ax.set_title('All Classifiers — PR Curves (CV)')
ax.legend(fontsize=7.5, loc='upper right')
ax.set_xlim([0, 1]); ax.set_ylim([0, 1])

# (0,1) F1 vs threshold (XGB only; naive shown as horizontal line)
ax = _axes_pr_cv[0, 1]
for name, probs, color, ls in _xgb_cv_clfs:
    _sc = _mms(probs)
    prec, rec, thresh = precision_recall_curve(_y_cv_pooled, _sc)
    f1 = 2 * prec[:-1] * rec[:-1] / (prec[:-1] + rec[:-1] + 1e-9)
    bi = np.argmax(f1)
    ax.plot(thresh, f1, color=color, ls=ls, lw=1.5, alpha=0.8,
            label=f'{name.split("(")[0].strip()} (F1={f1[bi]:.3f})')
    ax.scatter(thresh[bi], f1[bi], color=color, s=40, zorder=5)
ax.axhline(_naive_f1_cv, color='gray', ls='--', lw=1.2,
           label=f'Naive: DAM>$100 F1={_naive_f1_cv:.3f} (fixed point)')
ax.set_xlabel('Threshold (rescaled to [0, 1])'); ax.set_ylabel('F1 Score')
ax.set_title('F1 vs Threshold — CV')
ax.legend(fontsize=7.5, loc='upper right')
ax.set_xlim([0, 1])

# (1,0) PR-AUC bar chart — continuous scores only
ax = _axes_pr_cv[1, 0]
_prauc_names  = ['DAM\ncontinuous', 'XGB\nClf v1', 'XGB\nClf v2', 'XGB\nClf v3']
_prauc_vals   = [_dam_ap_cv] + [average_precision_score(_y_cv_pooled, p) for _, p, _, _ in _xgb_cv_clfs]
_prauc_colors = ['gray', 'tab:blue', 'tab:green', 'tab:red']
bars = ax.bar(_prauc_names, _prauc_vals, color=_prauc_colors, alpha=0.85, edgecolor='gray')
ax.axhline(_cv_baseline, color='black', ls=':', lw=1, label=f'Random ({_cv_baseline:.3f})')
for bar, val in zip(bars, _prauc_vals):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.005, f'{val:.3f}',
            ha='center', fontsize=9, fontweight='bold')
ax.set_ylabel('PR-AUC'); ax.set_title('PR-AUC Comparison — CV\n(continuous scores only)')
ax.legend(fontsize=8)

# (1,1) Per-fold F1 grouped bar chart — all 4 models × 3 folds
ax = _axes_pr_cv[1, 1]
_fold_names   = ['2022', '2023', '2024']
_models_pf    = ['Naive\nDAM>$100', 'XGB\nClf v1', 'XGB\nClf v2', 'XGB\nClf v3']
_colors_pf    = ['gray', 'tab:blue', 'tab:green', 'tab:red']
_f1_per_fold  = [
    _clf_cv_f1['naive'],
    _clf_cv_f1['v1'],
    _clf_cv_f1['v2'],
    _clf_cv_f1['v3'],
]
_n_models = len(_models_pf)
_n_folds  = len(_fold_names)
_x = np.arange(_n_folds)
_width = 0.18
_offsets = np.linspace(-1.5*_width, 1.5*_width, _n_models)
for i, (label, color, f1s) in enumerate(zip(_models_pf, _colors_pf, _f1_per_fold)):
    _bars = ax.bar(_x + _offsets[i], f1s, _width, label=label,
                   color=color, alpha=0.85, edgecolor='gray')
    for bar, val in zip(_bars, f1s):
        ax.text(bar.get_x() + bar.get_width()/2, val + 0.008, f'{val:.2f}',
                ha='center', va='bottom', fontsize=7, rotation=90)
ax.set_xticks(_x); ax.set_xticklabels(_fold_names)
ax.set_xlabel('Validation Fold'); ax.set_ylabel('F1 Score')
ax.set_title('Per-Fold F1 — CV\n(naive @ $100, XGB @ optimal threshold)')
ax.legend(fontsize=7.5, loc='upper right')
ax.set_ylim([0, ax.get_ylim()[1] * 1.15])

plt.tight_layout()
plt.savefig('figures/modeling/pr_curve_comparison_cv.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved figures/modeling/pr_curve_comparison_cv.png")

# ── Print CV metrics summary ──────────────────────────────────────────────────
print(f"\n{'Model':<35} {'PR-AUC':>7} {'F1':>6} {'Prec':>6} {'Rec':>6} {'Thresh':>7}")
print("-" * 68)
_naive_f1_mean = float(np.mean(_clf_cv_f1['naive']))
print(f"{'Naive: DAM > $100 (fixed)':<35} {'—':>7} {_naive_f1_mean:>6.3f} {'(per-fold mean)':>20}")
print(f"{'DAM continuous score':<35} {_dam_ap_cv:>7.3f} {'—':>6} {'—':>6} {'—':>6} {'swept':>7}")
for (name, _prob, _col), k in zip(_cv_models, ('v1', 'v2', 'v3')):
    _prauc_m = float(np.mean(_clf_cv_prauc[k]))
    _f1_m    = float(np.mean(_clf_cv_f1[k]))
    print(f"{name:<35} {_prauc_m:>7.3f} {_f1_m:>6.3f} {'(per-fold means)':>20}")
print(f"\nPooled CV: {_y_cv_pooled.sum()} spikes / {len(_y_cv_pooled)} hours "
      f"({_y_cv_pooled.mean():.2%})")
print(f"\nPer-fold F1:")
print(f"  {'Model':<20} {'2022':>6} {'2023':>6} {'2024':>6} {'Mean':>6}")
print(f"  {'-'*44}")
for label, f1s in zip(['Naive DAM>$100', 'XGB Clf v1', 'XGB Clf v2', 'XGB Clf v3'], _f1_per_fold):
    print(f"  {label:<20} {f1s[0]:>6.3f} {f1s[1]:>6.3f} {f1s[2]:>6.3f} {np.mean(f1s):>6.3f}")

/var/folders/vf/16_9fg814l76_gcv900gxyzc0000gn/T/ipykernel_87817/3500377547.py:102: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Saved figures/modeling/classification_diagnostics_cv.png


Saved figures/modeling/pr_curve_comparison_cv.png

Model                                PR-AUC     F1   Prec    Rec  Thresh
--------------------------------------------------------------------
Naive: DAM > $100 (fixed)                 —  0.471      (per-fold mean)
DAM continuous score                  0.473      —      —      —   swept
XGB Clf v1 (21 feat)                  0.363  0.433     (per-fold means)
XGB Clf v2 (29 feat)                  0.359  0.438     (per-fold means)
XGB Clf v3 (31 feat)                  0.351  0.426     (per-fold means)

Pooled CV: 1149 spikes / 26304 hours (4.37%)

Per-fold F1:
  Model                  2022   2023   2024   Mean
  --------------------------------------------
  Naive DAM>$100        0.467  0.518  0.428  0.471
  XGB Clf v1            0.314  0.547  0.438  0.433
  XGB Clf v2            0.319  0.542  0.452  0.438
  XGB Clf v3            0.301  0.546  0.430  0.426


/var/folders/vf/16_9fg814l76_gcv900gxyzc0000gn/T/ipykernel_87817/3500377547.py:196: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [5]:
# ── Naive DAM Classifier — spike if DAM price > $100/MWh ─────────────────────
# Simple baseline: if the day-ahead price already exceeds $100, flag as spike.
# No model needed — just a threshold on a single known feature.

from sklearn.metrics import (average_precision_score, f1_score,
                             precision_score, recall_score,
                             precision_recall_curve)

dam_naive_pred = (test['dam_price_houston'] > 100).astype(int)

# For PR-AUC we need a "score" — use DAM price directly as the ranking signal
dam_score = test['dam_price_houston'].fillna(0).values

prauc_naive = average_precision_score(y_te_clf, dam_score)
f1_naive    = f1_score(y_te_clf, dam_naive_pred)
prec_naive  = precision_score(y_te_clf, dam_naive_pred, zero_division=0)
rec_naive   = recall_score(y_te_clf, dam_naive_pred)

# XGB classifier at optimal threshold for comparison
_prec_pr, _rec_pr, _thresh_pr = precision_recall_curve(y_te_clf, prob_clf)
_f1_pr = 2 * _prec_pr[:-1] * _rec_pr[:-1] / (_prec_pr[:-1] + _rec_pr[:-1] + 1e-9)
_opt_idx = np.argmax(_f1_pr)
_opt_thresh = _thresh_pr[_opt_idx]
_pred_xgb = (prob_clf >= _opt_thresh).astype(int)

prauc_xgb = average_precision_score(y_te_clf, prob_clf)
f1_xgb    = f1_score(y_te_clf, _pred_xgb)
prec_xgb  = precision_score(y_te_clf, _pred_xgb)
rec_xgb   = recall_score(y_te_clf, _pred_xgb)

_xgb_label = f'XGB v3 (t={_opt_thresh:.3f})'
print("Spike Classifier Comparison — test 2025")
print(f"  Spike rate: {y_te_clf.mean():.1%} ({y_te_clf.sum()} / {len(y_te_clf)} hours)")
print(f"\n{'Model':<35} {'PR-AUC':>8} {'F1':>6} {'Prec':>6} {'Recall':>8}")
print("─" * 68)
print(f"{'Naive: DAM > $100':<35} {prauc_naive:>8.3f} {f1_naive:>6.3f} {prec_naive:>6.3f} {rec_naive:>8.3f}")
print(f"{_xgb_label:<35} {prauc_xgb:>8.3f} {f1_xgb:>6.3f} {prec_xgb:>6.3f} {rec_xgb:>8.3f}")
print(f"\nNaive DAM classifier flags {dam_naive_pred.sum()} hours as spikes")

Spike Classifier Comparison — test 2025
  Spike rate: 2.2% (196 / 8760 hours)

Model                                 PR-AUC     F1   Prec   Recall
────────────────────────────────────────────────────────────────────
Naive: DAM > $100                      0.296  0.288  0.362    0.240
XGB v3 (t=0.419)                       0.237  0.329  0.251    0.474

Naive DAM classifier flags 130 hours as spikes


In [6]:

# ── PR Curve Comparison — All Spike Classifiers (2×2, test 2025) ──────────────
# DAM continuous → full gray dashed curve (PR-AUC swept)
# Naive $100     → single gray star point (one operating point on DAM curve)

import pickle
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import (precision_recall_curve, average_precision_score,
                             f1_score, precision_score, recall_score)
from sklearn.preprocessing import minmax_scale
from pathlib import Path

PROC = Path('data/processed/ercot')

# ── Load classifiers and use stored feature names (avoids column order bugs) ──
with open(PROC / 'model_xgb_clf.pkl', 'rb') as f:
    m_clf_v1 = pickle.load(f)
with open(PROC / 'model_xgb_clf_v2.pkl', 'rb') as f:
    m_clf_v2 = pickle.load(f)

_fn_v1 = m_clf_v1.get_booster().feature_names  # authoritative order from pkl
_fn_v2 = m_clf_v2.get_booster().feature_names

# ── Compute scores (use DataFrame — XGBoost aligns by stored feature names) ──
prob_v1   = m_clf_v1.predict_proba(test[_fn_v1].fillna(0))[:, 1]
prob_v2   = m_clf_v2.predict_proba(test[_fn_v2].fillna(0))[:, 1]
dam_score = test['dam_price_houston'].fillna(0).values
pred_reg  = m_v3.predict(test[FEAT_V3])  # regression score as classifier

# ── Naive $100 fixed-threshold point ─────────────────────────────────────────
naive_pred_te = (dam_score >= 100).astype(int)
naive_prec_te = precision_score(y_te_clf, naive_pred_te, zero_division=0)
naive_rec_te  = recall_score(y_te_clf, naive_pred_te, zero_division=0)
naive_f1_te   = f1_score(y_te_clf, naive_pred_te, zero_division=0)

# ── Rescale to [0,1] for threshold-axis plots (PR-AUC invariant to scaling) ──
dam_score_01 = minmax_scale(dam_score)
pred_reg_01  = minmax_scale(pred_reg)

# XGB classifiers with continuous scores
xgb_classifiers = [
    ('XGB Classifier v1 (21 feat)', prob_v1,   prob_v1,      'tab:blue',   '-'),
    ('XGB Classifier v2 (29 feat)', prob_v2,   prob_v2,      'tab:green',  '-'),
    ('XGB Classifier v3 (31 feat)', prob_clf,  prob_clf,     'tab:red',    '-'),
    ('XGB Reg v3 as classifier',    pred_reg,  pred_reg_01,  'tab:orange', '-.'),
]

baseline_rate = y_te_clf.mean()
dam_ap_te = average_precision_score(y_te_clf, dam_score)

# ── 2×2 plot ──────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
fig.suptitle('Precision-Recall Curves — All Spike Classifiers (test 2025)', fontsize=13)

# (0,0) PR curves: DAM continuous curve + Naive $100 star point + XGB curves
ax = axes[0, 0]
dam_prec, dam_rec, _ = precision_recall_curve(y_te_clf, dam_score)
ax.plot(dam_rec, dam_prec, color='gray', ls='--', lw=2,
        label=f'DAM continuous score (PR-AUC={dam_ap_te:.3f})')
ax.scatter(naive_rec_te, naive_prec_te, color='gray', marker='*', s=250, zorder=6,
           label=f'Naive: DAM>$100 (F1={naive_f1_te:.3f}, point only)')
for name, raw, scaled, color, ls in xgb_classifiers:
    prec, rec, _ = precision_recall_curve(y_te_clf, raw)
    ap = average_precision_score(y_te_clf, raw)
    ax.plot(rec, prec, color=color, ls=ls, lw=2, label=f'{name} (PR-AUC={ap:.3f})')
ax.axhline(baseline_rate, color='black', ls=':', lw=1, label=f'Random ({baseline_rate:.3f})')
ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
ax.set_title('All Classifiers — PR Curves (test)')
ax.legend(fontsize=7.5, loc='upper right')
ax.set_xlim([0, 1]); ax.set_ylim([0, 1])

# (0,1) F1 vs threshold (XGB only; naive $100 shown as horizontal line)
ax = axes[0, 1]
for name, raw, scaled, color, ls in xgb_classifiers:
    prec, rec, thresh = precision_recall_curve(y_te_clf, scaled)
    f1 = 2 * prec[:-1] * rec[:-1] / (prec[:-1] + rec[:-1] + 1e-9)
    best_idx = np.argmax(f1)
    ax.plot(thresh, f1, color=color, ls=ls, lw=1.5, alpha=0.8,
            label=f'{name.split("(")[0].strip()} (F1={f1[best_idx]:.3f})')
    ax.scatter(thresh[best_idx], f1[best_idx], color=color, s=40, zorder=5)
ax.axhline(naive_f1_te, color='gray', ls='--', lw=1.2,
           label=f'Naive: DAM>$100 F1={naive_f1_te:.3f} (fixed point)')
ax.set_xlabel('Threshold (rescaled to [0, 1])'); ax.set_ylabel('F1 Score')
ax.set_title('F1 vs Threshold (all scores min-max scaled)')
ax.legend(fontsize=7.5, loc='upper right')
ax.set_xlim([0, 1])

# (1,0) PR-AUC bar chart — continuous scores only
ax = axes[1, 0]
prauc_names  = ['DAM\ncontinuous', 'XGB\nClf v1', 'XGB\nClf v2', 'XGB\nClf v3', 'XGB Reg\nas Clf']
prauc_vals   = [dam_ap_te] + [average_precision_score(y_te_clf, raw) for _, raw, _, _, _ in xgb_classifiers]
prauc_colors = ['gray', 'tab:blue', 'tab:green', 'tab:red', 'tab:orange']
bars = ax.bar(prauc_names, prauc_vals, color=prauc_colors, alpha=0.85, edgecolor='gray')
ax.axhline(baseline_rate, color='black', ls=':', lw=1, label=f'Random ({baseline_rate:.3f})')
for bar, val in zip(bars, prauc_vals):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.005, f'{val:.3f}',
            ha='center', fontsize=9, fontweight='bold')
ax.set_ylabel('PR-AUC'); ax.set_title('PR-AUC Comparison (test)\n(continuous scores only)')
ax.legend(fontsize=8)

# (1,1) F1 bar chart — naive $100 + XGB at optimal threshold
ax = axes[1, 1]
f1_names  = ['Naive\nDAM>$100', 'XGB\nClf v1', 'XGB\nClf v2', 'XGB\nClf v3', 'XGB Reg\nas Clf']
f1_colors = ['gray', 'tab:blue', 'tab:green', 'tab:red', 'tab:orange']
f1_vals   = [naive_f1_te]
for _, raw, _, _, _ in xgb_classifiers:
    prec, rec, _ = precision_recall_curve(y_te_clf, raw)
    f1 = 2 * prec[:-1] * rec[:-1] / (prec[:-1] + rec[:-1] + 1e-9)
    f1_vals.append(f1.max())
bars = ax.bar(f1_names, f1_vals, color=f1_colors, alpha=0.85, edgecolor='gray')
for bar, val in zip(bars, f1_vals):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.005, f'{val:.3f}',
            ha='center', fontsize=9, fontweight='bold')
ax.set_ylabel('F1'); ax.set_title('F1 Comparison (test)\n(naive @ $100, XGB @ optimal threshold)')

plt.tight_layout()
plt.savefig('figures/modeling/pr_curve_comparison_test.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Print metrics ─────────────────────────────────────────────────────────────
print(f"\n{'Model':<35} {'PR-AUC':>7} {'F1':>6} {'Prec':>6} {'Recall':>7}")
print("-" * 62)
print(f"{'Naive: DAM > $100 (fixed)':<35} {'—':>7} {naive_f1_te:>6.3f} "
      f"{naive_prec_te:>6.3f} {naive_rec_te:>7.3f}")
print(f"{'DAM continuous score':<35} {dam_ap_te:>7.3f} {'—':>6} {'—':>6} {'—':>7}")
for (name, raw, _, _, _), prauc, f1 in zip(xgb_classifiers, prauc_vals[1:], f1_vals[1:]):
    print(f"{name:<35} {prauc:>7.3f} {f1:>6.3f}")
print(f"\nSaved figures/modeling/pr_curve_comparison_test.png")


Model                                PR-AUC     F1   Prec  Recall
--------------------------------------------------------------
Naive: DAM > $100 (fixed)                 —  0.294  0.366   0.245
DAM continuous score                  0.296      —      —       —
XGB Classifier v1 (21 feat)           0.229  0.323
XGB Classifier v2 (29 feat)           0.237  0.317
XGB Classifier v3 (31 feat)           0.237  0.329
XGB Reg v3 as classifier              0.252  0.309

Saved figures/modeling/pr_curve_comparison_test.png


/var/folders/vf/16_9fg814l76_gcv900gxyzc0000gn/T/ipykernel_87817/2841919276.py:118: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [ ]:

# ── Test — Classification Diagnostics (XGB Classifier v3, Test Set 2025) ─
# Requires: y_te_clf, prob_clf, test, spike_mask from §8 setup.
# Counterpart: see CV version above (§9.2 CV cell, uses pooled walk-forward val).
from sklearn.metrics import (precision_recall_curve,
                             confusion_matrix, ConfusionMatrixDisplay)

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('XGBoost Classifier v3 — Classification Diagnostics (TEST SET 2025)', fontsize=12)

prec_pr, rec_pr, thresholds_pr = precision_recall_curve(y_te_clf, prob_clf)
f1_pr = 2 * prec_pr[:-1] * rec_pr[:-1] / (prec_pr[:-1] + rec_pr[:-1] + 1e-9)
opt_idx = np.argmax(f1_pr)
opt_thresh = thresholds_pr[opt_idx]

# 1. F1 vs threshold
axes[0, 0].plot(thresholds_pr, f1_pr, color='steelblue', lw=2)
axes[0, 0].axvline(opt_thresh, color='red', ls='--', lw=1.5,
                   label=f'Optimal t={opt_thresh:.3f}  F1={f1_pr[opt_idx]:.3f}')
axes[0, 0].set_xlabel('Threshold'); axes[0, 0].set_ylabel('F1 score')
axes[0, 0].set_title('F1 vs Threshold (Test)'); axes[0, 0].legend(fontsize=9)

# 2. PR curve
axes[0, 1].plot(rec_pr, prec_pr, color='coral', lw=2,
                label=f'PR-AUC = {average_precision_score(y_te_clf, prob_clf):.3f}')
axes[0, 1].axhline(y_te_clf.mean(), color='gray', ls='--', lw=1, label='Random baseline')
axes[0, 1].scatter(rec_pr[opt_idx], prec_pr[opt_idx], color='red', s=80, zorder=5,
                   label=f'Optimal t={opt_thresh:.3f}')
axes[0, 1].set_xlabel('Recall'); axes[0, 1].set_ylabel('Precision')
axes[0, 1].set_title('Precision-Recall Curve (Test)'); axes[0, 1].legend(fontsize=8)

# 3. Gains chart
sorted_idx = np.argsort(prob_clf)[::-1]
gains = np.cumsum(y_te_clf.values[sorted_idx]) / y_te_clf.sum()
pct_pop = np.arange(1, len(y_te_clf) + 1) / len(y_te_clf)
axes[0, 2].plot(pct_pop, gains, color='steelblue', lw=2, label='Model')
axes[0, 2].plot([0, 1], [0, 1], 'k--', lw=1, label='Random')
axes[0, 2].plot([0, y_te_clf.mean(), 1], [0, 1, 1], 'g--', lw=1, label='Perfect')
axes[0, 2].set_xlabel('% Population (by score)'); axes[0, 2].set_ylabel('% Spikes Captured')
axes[0, 2].set_title('Gains Chart (Test)'); axes[0, 2].legend(fontsize=8)

# 4. Lift chart
lift = gains / pct_pop
axes[1, 0].plot(pct_pop, lift, color='coral', lw=2)
axes[1, 0].axhline(1, color='k', ls='--', lw=1)
axes[1, 0].set_xlabel('% Population (by score)'); axes[1, 0].set_ylabel('Lift')
axes[1, 0].set_title('Lift Chart (Test)')

# 5. Confusion matrix at optimal threshold
y_pred_opt = (prob_clf >= opt_thresh).astype(int)
cm = confusion_matrix(y_te_clf, y_pred_opt)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Non-spike', 'Spike'])
disp.plot(ax=axes[1, 1], colorbar=False, cmap='Blues')
axes[1, 1].set_title(f'Confusion Matrix — Test (t={opt_thresh:.3f})')

# 6. FPR / FNR by month
test_clf_df = test.assign(prob=prob_clf, pred_clf=y_pred_opt, actual_clf=y_te_clf.values)
test_clf_df['month'] = test_clf_df.index.month
by_mo = (
    test_clf_df.groupby('month')[['pred_clf', 'actual_clf']]
    .apply(lambda g: pd.Series({
        'FPR': ((g['pred_clf']==1) & (g['actual_clf']==0)).sum() / max((g['actual_clf']==0).sum(), 1),
        'FNR': ((g['pred_clf']==0) & (g['actual_clf']==1)).sum() / max((g['actual_clf']==1).sum(), 1),
    }), include_groups=False)
    .reset_index()
)
month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
axes[1, 2].bar(by_mo['month'] - 0.2, by_mo['FPR'], width=0.35,
               color='steelblue', label='FPR')
axes[1, 2].bar(by_mo['month'] + 0.2, by_mo['FNR'], width=0.35,
               color='crimson', label='FNR (miss rate)')
axes[1, 2].set_xticks(range(1, 13))
axes[1, 2].set_xticklabels(month_names, rotation=45, ha='right')
axes[1, 2].set_ylabel('Rate'); axes[1, 2].set_title('FPR / FNR by Month (Test)')
axes[1, 2].legend(fontsize=8)

plt.tight_layout()
plt.savefig('figures/modeling/classification_diagnostics_test.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved classification_diagnostics_test.png")
print(f"Optimal threshold: {opt_thresh:.3f}  F1={f1_pr[opt_idx]:.3f}")
print(f"Confusion matrix at optimal threshold:\n{cm}")


Saved classification_diagnostics_test.png
Optimal threshold: 0.419  F1=0.329
Confusion matrix at optimal threshold:
[[8287  277]
 [ 103   93]]


/var/folders/vf/16_9fg814l76_gcv900gxyzc0000gn/T/ipykernel_87817/1560105804.py:78: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [8]:
# ── Block Bootstrap CI — Spike Classifier PR-AUC ─────────────────────────────
# Requires: y_te_clf, prob_clf from §8 setup.
# Block size = 24 (one day) to preserve intra-day dependence.

from sklearn.metrics import average_precision_score
import numpy as np
import matplotlib.pyplot as plt

def block_bootstrap_prauc(y_clf, y_prob, block_size=24, n_boot=2000, ci=95):
    """Block bootstrap CI for PR-AUC."""
    n = len(y_clf)
    alpha = (100 - ci) / 2
    rng = np.random.default_rng(42)
    prauc_boot = []

    for _ in range(n_boot):
        n_blocks = int(np.ceil(n / block_size))
        starts = rng.integers(0, n - block_size + 1, size=n_blocks)
        idx = np.concatenate([np.arange(s, s + block_size) for s in starts])[:n]
        yc, pr = y_clf[idx], y_prob[idx]
        if len(np.unique(yc)) == 2:
            prauc_boot.append(average_precision_score(yc, pr))

    prauc_pt = average_precision_score(y_clf, y_prob)
    q = np.percentile(prauc_boot, [alpha, 100 - alpha])
    prauc_ci = (2 * prauc_pt - q[1], 2 * prauc_pt - q[0])
    return prauc_pt, prauc_ci, prauc_boot

print("Running block bootstrap for PR-AUC (2000 iterations, block_size=24h)...")
prauc_pt, prauc_ci, prauc_boot = block_bootstrap_prauc(y_te_clf.values, prob_clf)
print("Done.")

print(f"\nClassifier v3 — 2025 test set (block bootstrap, 95% CI, block=24h):")
print(f"  PR-AUC = {prauc_pt:.3f}   95% CI: [{prauc_ci[0]:.3f}, {prauc_ci[1]:.3f}]")
print(f"\nNote: block_size=24h preserves intra-day autocorrelation.")

# ── Visualization ─────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(5, 4))
prauc_arr = np.array(prauc_boot)
ci_lo, ci_hi = np.percentile(prauc_arr, [2.5, 97.5])
ax.hist(prauc_arr, bins=50, color='darkorange', alpha=0.8, edgecolor='none')
ax.axvline(ci_lo, color='tomato', lw=1.5, ls='--', label=f'95% CI [{ci_lo:.3f}, {ci_hi:.3f}]')
ax.axvline(ci_hi, color='tomato', lw=1.5, ls='--')
ax.axvline(prauc_pt, color='navy', lw=2, label=f'Point {prauc_pt:.3f}')
ax.set_xlabel('PR-AUC'); ax.set_title('Classifier v3 — PR-AUC (block bootstrap)')
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig('figures/modeling/bootstrap_prauc_ci.png', dpi=150, bbox_inches='tight')
plt.show()
print("PR-AUC bootstrap CI plot saved.")

Running block bootstrap for PR-AUC (2000 iterations, block_size=24h)...


Done.

Classifier v3 — 2025 test set (block bootstrap, 95% CI, block=24h):
  PR-AUC = 0.237   95% CI: [0.151, 0.308]

Note: block_size=24h preserves intra-day autocorrelation.


PR-AUC bootstrap CI plot saved.


/var/folders/vf/16_9fg814l76_gcv900gxyzc0000gn/T/ipykernel_87817/1473673095.py:49: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Spike Classifier Calibration

Good ranking quality (high PR-AUC) does not guarantee that predicted probabilities are meaningful. A **reliability (calibration) diagram** checks whether predicted spike probabilities match observed spike frequencies. A perfectly calibrated classifier lies on the diagonal. Miscalibrated probabilities would make the threshold of 0.508 unreliable for operational use.

> *Note: Calibration analysis is not covered in the course reference notebooks. We include it here because the classifier threshold (0.508) is used for deployment decisions — if probabilities are overconfident, the threshold may need recalibration.*

In [ ]:

# ──  Spike Classifier Calibration ───────────────────────────────
# Requires: m_clf_v3 (XGBoost classifier), X_test_v3, y_clf_t (from Section 7.3 / 8)
# In Section 8, these correspond to: m_clf (loaded), X_te_clf, y_te_clf

from sklearn.calibration import CalibrationDisplay
from sklearn.metrics import brier_score_loss
import matplotlib.pyplot as plt
import numpy as np

# Get predicted probabilities from the v3 classifier
_proba = prob_clf   # m_clf.predict_proba(X_te_clf)[:, 1]

# Brier score (lower = better; 0 = perfect)
_brier = brier_score_loss(y_te_clf, _proba)
_brier_baseline = brier_score_loss(y_te_clf, np.full_like(_proba, y_te_clf.mean()))
_bss = 1 - _brier / _brier_baseline   # Brier Skill Score

fig, ax = plt.subplots(figsize=(7, 6))
CalibrationDisplay.from_predictions(
    y_te_clf, _proba,
    n_bins=10,
    ax=ax,
    name='XGBoost v3 classifier',
    color='steelblue'
)
ax.set_title(f'Spike Classifier Reliability Diagram\n'
             f'Brier={_brier:.4f} · Brier Skill Score={_bss:.3f} · Threshold=0.508')
ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Perfect calibration')
ax.legend()
plt.tight_layout()
plt.savefig('figures/modeling/spike_classifier_calibration.png', dpi=150)
plt.show()

print(f"Brier Score:       {_brier:.4f}  (baseline={_brier_baseline:.4f})")
print(f"Brier Skill Score: {_bss:.3f}  (>0 = better than climatology)")
print(f"\nCalibration interpretation:")
print(f"  If curve is above diagonal → model underestimates spike probability (conservative)")
print(f"  If curve is below diagonal → model overestimates spike probability (aggressive)")


Brier Score:       0.0294  (baseline=0.0219)
Brier Skill Score: -0.343  (>0 = better than climatology)

Calibration interpretation:
  If curve is above diagonal → model underestimates spike probability (conservative)
  If curve is below diagonal → model overestimates spike probability (aggressive)


/var/folders/vf/16_9fg814l76_gcv900gxyzc0000gn/T/ipykernel_87817/1810193944.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
